# 03 — Embeddings et recherche sémantique

Génération d'embeddings de documents via `sentence-transformers`
(modèle multilingue `paraphrase-multilingual-MiniLM-L12-v2`, optimisé pour le
français), stockage compatible `pgvector`, puis recherche par similarité
cosinus : requête en langage naturel → embedding → top-k documents les plus
proches.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from time import time

from utils import ARTIFACTS_DIR, save_embeddings, load_embeddings, load_documents_csv

df = load_documents_csv("documents_extraits.csv")
texts = df["texte_nettoye"].fillna("").astype(str).tolist()
ids = df["id"].tolist()
print(f"{len(texts)} documents à encoder.")

48 documents à encoder.


## 1. Chargement du modèle d'embeddings

Tentative de chargement de `sentence-transformers` (modèle
`paraphrase-multilingual-MiniLM-L12-v2`, téléchargé depuis Hugging Face). Si le
réseau ne permet pas ce téléchargement (environnement sans accès à
`huggingface.co`, ce qui est aussi le cas attendu sur le réseau interne 2M une
fois hors ligne), on bascule sur une méthode d'embedding locale de repli
(TF-IDF + réduction de dimension SVD) qui ne nécessite aucun téléchargement.

**En production**, le modèle `sentence-transformers` doit être téléchargé une
seule fois puis mis en cache dans l'image Docker du service `ged_ai_api/`
(étape de build), afin que le service tourne ensuite entièrement hors ligne —
voir `ged_ai_api/Dockerfile`.

In [2]:
EMBEDDING_MODE = None
model = None

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    EMBEDDING_MODE = "sentence-transformers"
    print("Modèle sentence-transformers chargé.")
except Exception as e:
    print("sentence-transformers indisponible dans cet environnement :", e)
    print("Bascule sur le mode de repli local (TF-IDF + SVD).")
    EMBEDDING_MODE = "tfidf-svd-fallback"

print("Mode d'embedding utilisé :", EMBEDDING_MODE)

sentence-transformers indisponible dans cet environnement : We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
Bascule sur le mode de repli local (TF-IDF + SVD).
Mode d'embedding utilisé : tfidf-svd-fallback


In [3]:
def encode_documents(texts):
    """Retourne une matrice d'embeddings normalisés (n_docs, dim)."""
    if EMBEDDING_MODE == "sentence-transformers":
        emb = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(emb, dtype="float32")

    # --- Mode de repli : TF-IDF + SVD, normalisé pour similarité cosinus ---
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize

    global _fallback_vectorizer, _fallback_svd
    _fallback_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=2000)
    X_tfidf = _fallback_vectorizer.fit_transform(texts)

    n_components = min(128, X_tfidf.shape[1] - 1, X_tfidf.shape[0] - 1)
    _fallback_svd = TruncatedSVD(n_components=max(n_components, 2), random_state=42)
    X_reduced = _fallback_svd.fit_transform(X_tfidf)
    return normalize(X_reduced).astype("float32")


def encode_query(query):
    if EMBEDDING_MODE == "sentence-transformers":
        return np.asarray(model.encode([query], normalize_embeddings=True), dtype="float32")[0]
    from sklearn.preprocessing import normalize
    q_tfidf = _fallback_vectorizer.transform([query])
    q_reduced = _fallback_svd.transform(q_tfidf)
    return normalize(q_reduced).astype("float32")[0]

## 2. Encodage du corpus

In [4]:
t0 = time()
embeddings = encode_documents(texts)
print(f"Embeddings générés en {time() - t0:.2f}s — forme : {embeddings.shape}")

Embeddings générés en 0.07s — forme : (48, 47)


## 3. Export (format compatible pgvector : vecteurs + métadonnées id)

In [5]:
paths = save_embeddings(embeddings, ids, basename="embeddings")
print("Embeddings exportés :")
for k, v in paths.items():
    print(" -", k, "->", v)

# Sauvegarde du mode utilisé, pour que le service FastAPI sache quel encodeur
# utiliser à l'inférence (sentence-transformers ou fallback tfidf-svd)
with open(os.path.join(ARTIFACTS_DIR, "embedding_mode.txt"), "w") as f:
    f.write(EMBEDDING_MODE)
print("Mode d'embedding sauvegardé :", EMBEDDING_MODE)

Embeddings exportés :
 - npy -> /home/claude/get-2m/ged-ai/artifacts/embeddings.npy
 - meta -> /home/claude/get-2m/ged-ai/artifacts/embeddings_meta.csv
Mode d'embedding sauvegardé : tfidf-svd-fallback


## 4. Fonction de recherche sémantique

Requête en langage naturel → embedding → similarité cosinus avec tous les
documents → top-k résultats.

In [6]:
def recherche_semantique(query: str, top_k: int = 3):
    emb, meta = load_embeddings("embeddings")
    q_vec = encode_query(query)

    # similarité cosinus (les vecteurs sont déjà normalisés)
    scores = emb @ q_vec
    top_idx = np.argsort(-scores)[:top_k]

    resultats = pd.DataFrame({
        "id": meta["id"].iloc[top_idx].values,
        "score_similarite": scores[top_idx],
    })
    return resultats

## 5. Test sur des requêtes réalistes

In [7]:
requetes_test = [
    "procédure sécurité incendie",
    "contrat prestataire technique",
    "évaluation annuelle des employés",
    "architecture du système de diffusion",
    "brief pour le journal télévisé",
]

for q in requetes_test:
    print(f"\nRequête : {q!r}")
    print(recherche_semantique(q, top_k=3).to_string(index=False))


Requête : 'procédure sécurité incendie'
                              id  score_similarite
  Procedures_internes_sample.pdf          0.796094
 Procedures_internes_sample.docx          0.795499
Procedures_internes_synth_03.txt          0.683538

Requête : 'contrat prestataire technique'
                   id  score_similarite
Contrats_synth_07.txt          0.803009
Contrats_synth_06.txt          0.670700
Contrats_synth_00.txt          0.669055

Requête : 'évaluation annuelle des employés'
             id  score_similarite
RH_synth_06.txt          0.758917
RH_synth_03.txt          0.742825
RH_synth_01.txt          0.398533

Requête : 'architecture du système de diffusion'
                    id  score_similarite
Technique_synth_02.txt          0.777780
Technique_synth_05.txt          0.764200
Technique_synth_06.txt          0.745016

Requête : 'brief pour le journal télévisé'
                    id  score_similarite
Editorial_synth_02.txt          0.702078
Editorial_synth_03.txt        

## Notes

- En mode `sentence-transformers`, la dimension des embeddings est celle du
  modèle `paraphrase-multilingual-MiniLM-L12-v2` (384) — à utiliser pour
  dimensionner la colonne `vector(384)` dans PostgreSQL/`pgvector`.
- En mode de repli (`tfidf-svd-fallback`), la dimension dépend du corpus
  (`min(128, ...)` ici) — **ce mode est une solution de secours pour
  l'exécution de ce notebook en environnement restreint, pas une recommandation
  de production**. En production, privilégier systématiquement
  `sentence-transformers` avec le modèle pré-téléchargé dans l'image Docker.
- Le fichier `embedding_mode.txt` permet au service FastAPI de savoir quel
  encodeur recharger au démarrage.